## Tracking of a Fish-position dataset optimised over a parameter grid, using likelihoods

In [6]:
## Importing data and converting to a track
from datetime import datetime, timedelta
import os
from ordered_set import OrderedSet
import pandas as pd
from stonesoup.types.detection import Clutter, Detection, TrueDetection
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState
from stonesoup.models.base_driver import NoiseCase
from stonesoup.models.driver import AlphaStableNSMDriver 
from stonesoup.models.transition.levy_linear import LevyLangevin, CombinedLinearLevyTransitionModel
from stonesoup.models.measurement.linear import LinearGaussian
import numpy as np
from scipy.stats import uniform
# Define Predictor, Resampler, and Updater
from stonesoup.predictor.particle import MarginalisedParticlePredictor
from stonesoup.resampler.particle import SystematicResampler
from stonesoup.types.track import Track
from stonesoup.updater.particle import MarginalisedParticleUpdater
# Particle Initialization
from scipy.stats import multivariate_normal
from stonesoup.types.numeric import Probability  # Similar to a float type
from stonesoup.types.state import MarginalisedParticleState
from stonesoup.types.array import CovarianceMatrices, StateVector, StateVectors
from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.models.transition.linear import ConstantVelocity, RandomWalk
from stonesoup.types.state import GaussianState
from stonesoup.updater.kalman import KalmanUpdater
from numpy.random      import default_rng

from stonesoup.types.state import GaussianState, MarginalisedParticleState
from stonesoup.types.array import StateVectors, CovarianceMatrices
from datetime import timedelta

In [7]:
############################################################
# 0.  Imports  +  global constants used everywhere
############################################################

# ---------- generic SMC settings ----------
num_particles          = 2000            # per-object MPF particles
P_D                    = 1         # detection prob.
KAPPA_Z                = 1e-5           # clutter spatial density
SEED                   = 1
rng                    = default_rng(SEED)
MAX_CLUTTER=0

# process only the first N lines of every CSV
optimise_T   = 100
filter_T     = 500
num_steps    = max(optimise_T, filter_T)
num_sigma2   = 1
sigma_e_factor = 0.05

# ---------- folders & file list ----------
folder = fr"TrackedDatasets\FishData\train"
files = [
    # rf"ZebraFish-01",
    rf"ZebraFish-02",
    # rf"ZebraFish-03",
    # rf"ZebraFish-04",
    ]

############################################################
# 1–3.  Load CSV  ✧  empirical σᵉ  ✧  detections
############################################################
datasets = {}                                      # name → info-dict
priors = {}                                     # file → (lp_prior , gp_prior)

for name in files:
    txt_path  = os.path.join(folder, name, "gt", "gt.txt")

    # ------------------------------------------------------------------
    # Load & keep only frame, id, 3-D position
    # ------------------------------------------------------------------
    cols = ["frame", "id", "x", "y", "z"]
    df   = pd.read_csv(txt_path, header=None, sep=",",
                       usecols=[0, 1, 2, 3, 4], names=cols)\
                .iloc[:num_steps]

    # timeline (1 kHz → ms indexes in MOT files)
    start_time = datetime.now()
    ts         = [start_time + timedelta(milliseconds=int(k))
                  for k in range(num_steps)]

    # ------------------------------------------------------------------
    # Build one GroundTruthPath per fish-ID
    # ------------------------------------------------------------------
    truths = []                                   # id → GroundTruthPath
    for fish_id, grp in df.groupby("id"):
        truth = GroundTruthPath()
        for k in range(num_steps):
            row = grp.iloc[min(k, len(grp)-1)]               # stay at last pose
            state = np.array([
            [row.x],   
            [row.y], 
            [row.z]
            ])
            truth.append(GroundTruthState(state_vector=state,
                                         timestamp=ts[k]))
        truths.append(truth)                                # keep ordering

    # Pooled stats for clutter
    all_xyz = df[["x", "y", "z"]].to_numpy(float)
    x_std, y_std, z_std = np.std(all_xyz, axis=0)

    # ------------------------------------------------------------------
    # Empirical σₑ from first fish’s Δ-positions (coarse but sufficient)
    # ------------------------------------------------------------------
    sigma2_e = sigma_e_factor*x_std**2
    if num_sigma2==1:
        sigma2_grid=[sigma2_e]
    else:
        sigma2_grid = np.logspace(np.log10(sigma2_e) - 2,
                              np.log10(sigma2_e), num_sigma2)
        
    # ─── measurement models (one per σ²) ──────────────────────────────
    meas_models = {
        s2: LinearGaussian(ndim_state=3, mapping=(0,1,2),
                           noise_covar=s2*np.eye(3))
        for s2 in sigma2_grid
    }
    
    # ------------------------------------------------------------------
    # Build detection list:   time-indexed *set*  (True + clutter)
    # Same list reused for every σ² so each grid-point sees identical data
    # ------------------------------------------------------------------
    all_measurements = {s2: [] for s2 in sigma2_grid}

    for s2 in sigma2_grid:
        for k, t in enumerate(ts):
            mset = set()

            # --- true detections (one per path) --------------------------
            for truth in truths:
                if np.random.rand() <= P_D:
                    meas = meas_models[s2].function(truth[k], noise=False)
                    #Simply change to False, and num in sigma_grid to >1 to discrete sum over measurement sigmas, 
                    # assuming the data are observations rather than the groundtruth

                    mset.add(TrueDetection(state_vector=meas,
                                            groundtruth_path=truth,
                                            timestamp=t,
                                            measurement_model=meas_models[s2]))

            # --- clutter -----------------------------------------------
            kappa_z = 0.5*(MAX_CLUTTER-1)/( (4*x_std)**3 )
            if MAX_CLUTTER:
                for _ in range(rng.integers(MAX_CLUTTER)):
                    x = uniform.rvs(np.mean(all_xyz[:,0])-2*x_std, 4*x_std, random_state=rng)
                    y = uniform.rvs(np.mean(all_xyz[:,1])-2*y_std, 4*y_std, random_state=rng)
                    z = uniform.rvs(np.mean(all_xyz[:,2])-2*z_std, 4*z_std, random_state=rng)
                    mset.add(Clutter(state_vector=np.array([[x],[y],[z]]),
                                    timestamp=t,
                                    measurement_model=meas_models[s2]))
            all_measurements[s2].append(mset)
    # ------------------------------------------------------------------
    # Store everything for later
    # ------------------------------------------------------------------
    data = dict(
        ts            = ts,
        meas_models   = meas_models,
        measurements  = all_measurements,
        groundtruths  = truths, # list[GroundTruthPath]
        sigma2_grid   = sigma2_grid,
        sigma2_e      = sigma2_e
    )

    datasets[name] = data
    print(f"⇒ Prepared {name} fish-datasets:", ", ".join(datasets))
    ############################################################
    # 4.  Prior states  (one Marginalised-Particle + one Gaussian per file)
    ############################################################
    

    # 4.  Priors  (one LP & one GP Track per truth per CSV file)
    # ===============================================
    priors[name]=[[],[]] #one list for lp, one for gp
    for truth in truths:
        mu0 = np.array(truth[0].state_vector.flatten())
        
        # variance for positions from empirical σe; same for velocities
        Sigma0  = np.diag([sigma2_e]*3)  
        # build (d, d, N) covariance stack more succinctly
        cov_stack = np.tile(Sigma0[:, :, None], (1, 1, num_particles))
        states=multivariate_normal.rvs(mu0,Sigma0,num_particles)
        t0 = ts[0] - timedelta(milliseconds=1)    

        #  –– particle prior
        lp = Track(MarginalisedParticleState(
            state_vector = StateVectors(states.T),
            covariance   = CovarianceMatrices(cov_stack),
            weight       = np.full(num_particles, 1/num_particles),
            timestamp    = t0
        ))
        priors[name][0].append(lp)
        
        #  –– Gaussian prior
        gp = Track(GaussianState(
            state_vector = mu0,
            covar        = Sigma0,
            timestamp    = t0
        ))
        
        priors[name][1].append(gp)

⇒ Prepared ZebraFish-02 fish-datasets: ZebraFish-02


In [8]:
############################################################
# 5.  Tiny helper : tolerant dictionary key look-up
############################################################
def close_key(dct, value, tol=1e-12):
    """Return the key in *dct* closest to *value* (within tol)."""
    keys = np.fromiter(dct, float)
    k    = keys[np.argmin(np.abs(keys - value))]
    if abs(k - value) > tol:
        raise KeyError(value)
    return float(k)

In [9]:
############################################################
# 6.  Cached builders  (transition-model ➜ predictor + updaters)
############################################################
from functools import lru_cache

from stonesoup.measures.state import Mahalanobis

# — Gaussian side —
from stonesoup.dataassociator.neighbour import GlobalNearestNeighbour
from stonesoup.hypothesiser.distance import DistanceHypothesiser
from stonesoup.models.transition.linear import (ConstantVelocity,
    RandomWalk, OrnsteinUhlenbeck, CombinedLinearGaussianTransitionModel)
from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.updater.kalman   import KalmanUpdater

# — Lévy / particle side —
from stonesoup.models.driver               import AlphaStableNSMDriver
from stonesoup.models.transition.levy_linear import (
    LevyRandomWalk, LevyConstantVelocity,LevyLangevin, CombinedLinearLevyTransitionModel)
from stonesoup.predictor.particle          import MarginalisedParticlePredictor
from stonesoup.updater.particle            import MarginalisedParticleUpdater
from stonesoup.resampler.particle          import SystematicResampler

RESAMPLER = SystematicResampler()           # single shared instance


@lru_cache(maxsize=None)
def build_gp(name: str, sigma_w2: float):
    """
    Return (predictor, {σe² : updater}) for the Gaussian model attached
    to *name*.  • OU in 2-D (hidden velocity)  • RW in 2- or 3-D otherwise.
    """

    # --- choose transition model ----------------------------------
    tm = CombinedLinearGaussianTransitionModel(
                    [RandomWalk(sigma_w2**0.5)]*3)

    predictor = KalmanPredictor(tm)
    updaters  = {s2: KalmanUpdater(mdl)
                 for s2, mdl in datasets[name]['meas_models'].items()}
    data_associators = {s2: GlobalNearestNeighbour(
                                                DistanceHypothesiser(predictor, KalmanUpdater(mdl),
                                                measure=Mahalanobis(), 
                                                missed_distance=25)) 
                                                for s2, mdl in datasets[name]['meas_models'].items()}

    return data_associators, updaters


@lru_cache(maxsize=None)
def build_lp(name: str, sigma_w2: float, alpha: float):
    """
    Return (predictor, {σe² : updater}) for the Lévy (MPF) model
    attached to *name*.  • Lévy-Langevin in 2-D with vel,
    • Lévy RW in 2- or 3-D otherwise.
    """
    driver = AlphaStableNSMDriver(mu_W=0.0,
                                  sigma_W2=sigma_w2,
                                  c=10.0,
                                  alpha=alpha,
                                  noise_case=NoiseCase(2))

    tm = CombinedLinearLevyTransitionModel(
                    [LevyRandomWalk(driver, noise_diff_coeff=sigma_w2**0.5)]*3)
        
    predictor = MarginalisedParticlePredictor(tm)
    updaters  = {s2: MarginalisedParticleUpdater(mdl, RESAMPLER)
                 for s2, mdl in datasets[name]['meas_models'].items()}
    data_associators = {s2: GlobalNearestNeighbour(
                                            DistanceHypothesiser(predictor, MarginalisedParticleUpdater(mdl, RESAMPLER),
                                            measure=Mahalanobis(), 
                                            missed_distance=25)) 
                                            for s2, mdl in datasets[name]['meas_models'].items()}
    return data_associators, updaters


In [ ]:
###################################################################
# 7–9.  Per-file Nelder–Mead search for (σw², α])            #
###################################################################
import copy
from scipy.optimize import minimize
from scipy.special  import logsumexp
from stonesoup.types.hypothesis import SingleHypothesis
from copy import deepcopy
from pathlib import Path
from datetime import timedelta

from stonesoup.plotter import Plotterly, AnimatedPlotterly, Dimension
from stonesoup.smoother.particle import MarginalisedKalmanSmoother
from stonesoup.types.hypothesis import SingleHypothesis
from stonesoup.updater.kalman      import KalmanUpdater
from stonesoup.updater.particle    import MarginalisedParticleUpdater
from stonesoup.resampler.particle  import SystematicResampler
# ────────────────────────────────────────────────────────────────
#  helper: log-likelihood for ONE file & ONE hyper-parameter set
# ────────────────────────────────────────────────────────────────
def _filter_loglike(name, params, model='gp'):
    """Log p(y | θ) with σe² integrated-out (Monte-Carlo for LP)."""
    if model == 'gp':
        σw2 = params
        associators, updaters, = build_gp(name, σw2)
        base_tracks     = priors[name][1]          # GP prior Tracks
    else:
        σw2, α = params
        associators, updaters = build_lp(name, σw2, α)
        base_tracks     = priors[name][0]          # LP prior Tracks

    ll_total = 0.0
    for s2, all_meas in datasets[name]['measurements'].items():        
        tracks   = [copy.deepcopy(base_track) for base_track in base_tracks]         # fresh copy
        associator=associators[s2]
        updater = updaters[s2]
        
        for t, dets in enumerate(all_meas[:optimise_T]):
            timestamp=datasets[name]['ts'][t]

            hypotheses= associator.associate(tracks=tracks,
                                            detections=dets,
                                            timestamp=timestamp)
            for track in tracks:
                hypo=hypotheses[track]
                if hypo.measurement:
                    det=hypo.measurement
                    post       = updater.update(hypo)
                    track.append(post)

                    mp   = hypo.measurement_prediction
                    y    = det.state_vector.flatten()
                    if model == 'gp':                     # single Gaussian
                        μ = mp.state_vector.flatten()
                        Σ = mp.covar
                        diff = y - μ
                        Σinv = np.linalg.inv(Σ)
                        maha = diff @ Σinv @ diff
                        d    = y.size
                        sign, logdet = np.linalg.slogdet(Σ)
                        ll_total += -0.5*(maha + d*np.log(2*np.pi) + logdet)
                    else:                                 # particle mixture
                        means = mp.state_vector           # (d,N)
                        covs  = mp.covariance             # (d,d,N)
                        d, N  = means.shape
                        ll_arr = np.empty(N)
                        for j in range(N):
                            diff  = y - means[:, j]
                            Σinv  = np.linalg.inv(covs[:, :, j])
                            maha  = diff @ Σinv @ diff
                            sign, logdet = np.linalg.slogdet(covs[:, :, j])
                            ll_arr[j] = -0.5*(maha + d*np.log(2*np.pi) + logdet)
                        ll_total += logsumexp(ll_arr) - np.log(N)
                else:
                    track.append(hypo.prediction)
                    ll_total += np.log(1-P_D)
    return ll_total


# ────────────────────────────────────────────────────────────────
#  dictionaries to hold arg-max results
# ────────────────────────────────────────────────────────────────
Best_gp_params = {}      # file → (σw2 , θ)
Best_lp_params = {}      # file → (σw2 , θ , α)

print("⏳  Optimising hyper-parameters (per file) …")

for name in datasets.keys():

    # --- convenience sigma2_e for initialisation ------------------
    init_σw2 = 1                                   # reasonable start based on data
    # init_θ  = 0.1
    init_α  = 1.4
    
    # -------------- objective wrappers for SciPy -----------------
    def obj_gp(x):
        σw2 = np.exp(x[0])
        # θ   = 0.05+ 0.45/ (1.0 + np.exp(-x[1]))                # map ℝ → (0.05,0.5)
        print('iteration')
        return -_filter_loglike(name, (σw2), model='gp')

    def obj_lp(x):
        σw2  = np.exp(x[0])
        # θ    = 0.05+ 0.45/ (1.0 + np.exp(-x[1]))                # map ℝ → (0.05,0.5)
        α    = 0.5 + 1.4/(1+np.exp(-x[1]))               # (0.5,1.9)
        print('iteration')
        if abs(α-1.0) < 1e-3:   # avoid α=1 exactly
            α += 0.05
        return -_filter_loglike(name, (σw2, α), model='lp')

    # ---------------- Nelder–Mead (Gaussian) ---------------------
    res_gp = minimize(obj_gp,
                      x0=[np.log(init_σw2)],
                      method='Nelder-Mead',
                      options={'maxiter': 20, 'disp': False})

    σw_best_gp = np.sqrt(np.exp(res_gp.x[0]))
    # θ_best_gp  = 0.05+ 0.45/ (1.0+ np.exp(-res_gp.x[1]))
    Best_gp_params[name] = (σw_best_gp) #, θ_best_gp)

    print(f"  • {name:<24}   GP→ σw={σw_best_gp:.6f}  ") #, θ={θ_best_gp:.6f}

    # ---------------- Nelder–Mead (Lévy) -------------------------
    res_lp = minimize(obj_lp,
                      x0=[np.log(init_σw2), init_α],
                      method='Nelder-Mead',
                      options={'maxiter': 20, 'disp': False})

    σw_best_lp = np.sqrt(np.exp(res_lp.x[0]))
    # θ_best_lp  = 0.05+ 0.45/ (1.0+ np.exp(-res_lp.x[1]))
    α_best_lp  = 0.5 + 1.4/(1+np.exp(-res_lp.x[1]))
    if abs(α_best_lp-1.0) < 1e-3:
        α_best_lp += 0.05
    Best_lp_params[name] = (σw_best_lp,  α_best_lp) #θ_best_lp,

    print(f"  • {name:<24}   Lévy→ σw={σw_best_lp:.6f}, α={α_best_lp:.6f}") # θ={θ_best_lp:.6f},
    
    ############################################################
    # 10–12.  Optimal-parameter filtering  ➜  HTML plots
    ############################################################
    Optimal_lp_tracks, Optimal_gp_tracks = {}, {}

    out_root = Path(r"C:\Users\joesb\OneDrive\Documents\Cambridge\IIB"
                    r"\PROJECT- Implementation of N-G TAs in SS framework\RandomFishPlots")
    out_root.mkdir(parents=True, exist_ok=True)

    def _pretty(fig, xlab, ylab):
        fig.update_layout(width=1200, height=640, plot_bgcolor="white",
                        xaxis=dict(showgrid=True, gridcolor="lightgray",
                                    title=dict(text=xlab, font=dict(size=20))),
                        yaxis=dict(showgrid=True, gridcolor="lightgray",
                                    title=dict(text=ylab, font=dict(size=20))),
                        legend=dict(font=dict(size=15),
                                    bordercolor="Black", borderwidth=2))
    data = datasets[name]
    truths      = data['groundtruths']
    ts          = data['ts']

    # ---- 1.  pick one σe² slice (empirical nearest) ------------------

    # σe2_emp      = np.min(data['sigma2_grid'])
    σe2_emp      = np.median(data['sigma2_grid'])
    σe2_plot     = close_key(data['meas_models'], σe2_emp)
    meas_model   = data['meas_models'][σe2_plot]
    all_meas     = data['measurements'][σe2_plot]

    gp_upd_plot  = KalmanUpdater(meas_model)
    lp_upd_plot  = MarginalisedParticleUpdater(meas_model, SystematicResampler())

    # ---- 2.  build predictors from the *best* θ, α, σw ---------------
    σw2_gp            = Best_gp_params[name]
    σw2_lp, α_lp         = Best_lp_params[name]

    gp_associator_opt, gp_updater_opt = build_gp(name, σw2_gp)
    lp_associator_opt, lp_updater_opt= build_lp(name, σw2_lp, α_lp)

    # fresh priors
    lp_tracks = deepcopy(priors[name][0])
    gp_tracks = deepcopy(priors[name][1])

    # ---- 3.  one-pass filtering over *all* detections ----------------
    for t, dets in enumerate(all_meas[:filter_T]):
        timestamp=ts[t]

        lp_hypotheses=lp_associator_opt[σe2_plot].associate(tracks=lp_tracks,
                                            detections=dets,
                                            timestamp=timestamp)
        for lp_track in lp_tracks:
            # Lévy
            lp_hypo= lp_hypotheses[lp_track]
            if lp_hypo.measurement:
                lp_post       = lp_updater_opt[σe2_plot].update(lp_hypo)
                lp_track.append(lp_post)
            else:
                lp_track.append(lp_hypo.prediction)

        gp_hypotheses=gp_associator_opt[σe2_plot].associate(tracks=gp_tracks,
                                            detections=dets,
                                            timestamp=timestamp)
        for gp_track in gp_tracks:
            # Gaussian
            gp_hypo= gp_hypotheses[gp_track]
            if gp_hypo.measurement:
                gp_post       = gp_updater_opt[σe2_plot].update(gp_hypo)
                gp_track.append(gp_post)
            else:
                gp_track.append(gp_hypo.prediction)
    print(name,"track filtered")

    Optimal_lp_tracks[name], Optimal_gp_tracks[name] = lp_tracks, gp_tracks

    colors=['#00CC96','#FFA15A','#B6E880','#AB63FA','red']
    # -----------------------------------------------------------------
    # 4-A.  1-D x-coordinate panel
    # -----------------------------------------------------------------
    p1 = Plotterly(dimension=Dimension.ONE)
    # p1.plot_measurements(all_meas, [0], label="Measurements",marker=dict(size=8))

    RTSsmoother=MarginalisedKalmanSmoother()
    RTS_tracks=[]
    for i, truth in enumerate(truths):
        color = colors[i]
        lp_tracks[i]=Track(lp_tracks[i][1:]) #[1:] means we exclude prior state (isn't a prediction or a post)
        gp_tracks[i]=Track(gp_tracks[i][1:])
        p1.plot_ground_truths(truth, [0], truths_label=f"Observed {i} gt",
                    line=dict(width=3,color=color,dash='solid'))
        p1.plot_tracks(gp_tracks[i], [0], track_label=f"Gaussian {i}", uncertainty=True,
                    mode='lines',line=dict(width=3, color=color,dash=f"dash"))
        p1.plot_tracks(lp_tracks[i], [0], track_label=f"Lévy {i}", uncertainty=True,
                    mode='lines', line=dict(width=3, color=color,dash='dot'))
        RTStrack=RTSsmoother.smooth(lp_tracks[i]) 
        RTS_tracks.append(RTStrack)
        p1.plot_tracks(RTS_tracks[i], [0], track_label=f"Lévy {i} RTS", uncertainty=True,
                    mode='lines', line=dict(width=3, color=color, dash='longdashdot'))

    print(name,"track plotted")
    
    _pretty(p1.fig, "Time","x-position")
    p1.fig.write_html(out_root / f"{name}_1D.html")

    # -----------------------------------------------------------------
    # 4-B.  2-D / 3-D animated trajectory (skip if pure-1D data)
    # -----------------------------------------------------------------
    times = ts
    mapping  = [0,2]
    
    p2 = AnimatedPlotterly(timesteps=times,tail_length=0.5)
    # p2.plot_measurements(all_meas, mapping, label="Measurements",marker=dict(size=8))
    for i, truth in enumerate(truths):
        color = colors[i]
        p2.plot_ground_truths(truth, mapping, truths_label=f"Observed {i} gt",
                    line=dict(width=3,color=color,dash='solid'))
        p2.plot_tracks(gp_tracks[i], mapping, track_label=f"Gaussian {i}", uncertainty=False,
                    mode='lines',line=dict(width=3, color=color,dash="dash"))
        p2.plot_tracks(lp_tracks[i], mapping, track_label=f"Lévy {i}", uncertainty=False,
                    mode='lines',line=dict(width=3, color=color,dash='dot'))
        p2.plot_tracks(RTS_tracks[i], mapping, track_label=f"Lévy {i} RTS", uncertainty=False,
                    mode='lines',line=dict(width=3, color=color, dash='longdashdot'))
    _pretty(p2.fig, "x-position","y-position")
    p2.fig.write_html(out_root / f"{name}_2Danimated.html")

    p2 = Plotterly()
    # p2.plot_measurements(all_meas, mapping, label="Measurements",marker=dict(size=8))
    for i, truth in enumerate(truths):
        color = colors[i]
        p2.plot_ground_truths(truth, mapping, truths_label=f"Observed {i} gt",
                    line=dict(width=3,color=color,dash='solid'))
        p2.plot_tracks(gp_tracks[i], mapping, track_label=f"Gaussian {i}", uncertainty=False,
                    mode='lines',line=dict(width=3, color=color,dash="dash"))
        p2.plot_tracks(lp_tracks[i], mapping, track_label=f"Lévy {i}", uncertainty=False,
                    mode='lines',line=dict(width=3, color=color,dash='dot'))
        p2.plot_tracks(RTS_tracks[i], mapping, track_label=f"Lévy {i} RTS", uncertainty=False,
                    mode='lines',line=dict(width=3, color=color, dash='longdashdot'))
    _pretty(p2.fig, "x-position","y-position")
    p2.fig.write_html(out_root / f"{name}_2D.html")
    datasets[name]=None #wipe to try save storage

print("🎉  Filtering complete – HTML plots written to", out_root)


⏳  Optimising hyper-parameters (per file) …
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
  • ZebraFish-02               GP→ σw=23.756947  
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
iteration
